# 10 — LLM Explanations

LLMs can generate explanations in natural language — but are these explanations faithful to what the model actually computes?

## Self-Explanation and CoT Faithfulness

**Self-explanation** refers to an LLM explaining its own outputs. An LLM asked 'why did you say X?' may produce a plausible-sounding rationale that is *post-hoc rationalisation* rather than an accurate account of its computation.

**Chain-of-thought (CoT) as explanation**: CoT prompting elicits reasoning traces before the final answer. These traces look like explanations, but research (Lanham et al., 2023; Turpin et al., 2023) shows:
- Unfaithfulness: LLMs reach the same answer with or without the chain-of-thought
- Sycophancy: CoT reflects the desired answer rather than the reasoning that led to it
- Counterfactual tests: add a biasing hint (e.g., 'I think the answer is A') and the CoT changes to support A even when the correct answer is B

**Faithfulness tests for CoT**:
1. *Counterfactual*: does the CoT change when the answer changes?
2. *Truncation*: does truncating the CoT change the final answer?
3. *Paraphrase consistency*: does re-ordering the CoT change the answer?
4. *Biasing hint*: does inserting an incorrect answer into the prompt corrupt the reasoning chain?

In [ ]:
# CoT faithfulness audit pipeline (simulated)
import numpy as np

# Simulate an LLM that answers yes/no questions with CoT
# We inject a 'biasing hint' and measure if the CoT changes

class SimulatedLLM:
    """Simulates an LLM with configurable faithfulness."""
    def __init__(self, faithfulness=0.7, seed=42):
        self.faithfulness = faithfulness
        np.random.seed(seed)

    def answer(self, question, hint=None):
        # True answer based on question content
        true_answer = 'yes' if hash(question) % 2 == 0 else 'no'

        # If hint is provided, model may be swayed
        if hint and np.random.random() > self.faithfulness:
            final_answer = hint  # sycophantic: follows hint
            reasoning = f'Thinking about this... {hint} seems right because the question implies it.'
        else:
            final_answer = true_answer
            reasoning = f'Let me reason through this step by step. Based on the evidence, {true_answer} is correct.'

        return {'answer': final_answer, 'reasoning': reasoning, 'true_answer': true_answer}

# Audit: biasing hint test
questions = [
    'Is Paris the capital of France?',
    'Is 7 a prime number?',
    'Does water freeze at 100 degrees Celsius?',
    'Is Python an interpreted language?',
    'Do birds have four legs?',
]

llm = SimulatedLLM(faithfulness=0.6)

print('Biasing hint faithfulness test:')
print(f'{"Question":<45} {"True":>6} {"Clean":>6} {"Biased":>7} {"Faithful":>9}')
n_faithful = 0
for q in questions:
    clean = llm.answer(q)
    # Inject wrong hint
    wrong_hint = 'no' if clean['true_answer'] == 'yes' else 'yes'
    biased = llm.answer(q, hint=wrong_hint)
    faithful = clean['answer'] == biased['answer']  # answer unchanged despite hint
    if faithful: n_faithful += 1
    print(f'{q[:44]:<45} {clean["true_answer"]:>6} {clean["answer"]:>6} {biased["answer"]:>7} {str(faithful):>9}')

print(f'\nFaithfulness rate: {n_faithful}/{len(questions)} = {n_faithful/len(questions):.1%}')

In [ ]:
# Counterfactual consistency test
print('Counterfactual consistency test:')
print('(Does the CoT change when we ask for the opposite answer?)\n')

test_q = 'Is 7 a prime number?'
r1 = llm.answer(test_q)
r2 = llm.answer(test_q, hint='no')  # force opposite

print(f'Without hint: answer={r1["answer"]}')
print(f'  Reasoning: {r1["reasoning"]}')
print(f'\nWith hint=no: answer={r2["answer"]}')
print(f'  Reasoning: {r2["reasoning"]}')

# Automated consistency score: does answer change when CoT is truncated?
def truncation_consistency(llm, question, n_trials=20):
    """If truncating CoT changes the answer, the CoT was load-bearing (good)."""
    base = llm.answer(question)
    full_answer = base['answer']

    # Simulate truncated CoT (model re-answers without prior reasoning)
    changes = 0
    for _ in range(n_trials):
        truncated = llm.answer(question)  # fresh call
        if truncated['answer'] != full_answer:
            changes += 1
    return 1 - changes / n_trials  # high = consistent

print('\nAnswer consistency across repeated calls (high = stable):')
for q in questions:
    consistency = truncation_consistency(llm, q)
    print(f'  [{consistency:.2f}] {q[:60]}')

In [ ]:
# Faithfulness scorecard
import matplotlib.pyplot as plt

metrics = {
    'Biasing hint resistance': 0.6,
    'Answer consistency': 0.75,
    'Counterfactual sensitivity': 0.55,
    'Self-calibration': 0.65,
    'Verbalized conf. accuracy': 0.58,
}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['green' if v > 0.7 else 'orange' if v > 0.5 else 'red' for v in metrics.values()]
ax.barh(list(metrics.keys()), list(metrics.values()), color=colors, alpha=0.8)
ax.axvline(0.7, color='green', linestyle='--', label='Good threshold')
ax.axvline(0.5, color='orange', linestyle='--', label='Acceptable threshold')
ax.set_xlim(0, 1)
ax.set_xlabel('Score')
ax.set_title('LLM Explanation Faithfulness Scorecard')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/llm_faithfulness_scorecard.png', dpi=100, bbox_inches='tight')
plt.show()
print('LLM faithfulness scorecard saved')